This file consists of code to load the video and looping through its frames

Imports:

In [1]:
%matplotlib tk
import cv2
import os
import sys
from ultralytics import YOLO
from utils import VideoProcessor, BGRHandler
import numpy as np
import matplotlib.pyplot as plt

Loading the video

In [2]:
#when loading a video we want to use videocapture
cap = cv2.VideoCapture(r'input-videos/08fd33_4.mp4')

# Get the default frame width and height this is important for when we write the output video
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

# Define the codec and create VideoWriter object
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter('output.mp4', fourcc, 20.0, (frame_width, frame_height))
#the model we are using which is the best.pt model that we trained on our personal dataset
model = YOLO("train64/weights/best.pt")
#this is where we set the class colors dictionary for our classes
class_colors ={0: (0, 0, 255), 1: (0, 255, 0), 2: (255, 0, 0), 3:(0, 255, 255), 4:(255, 255, 0)}
#we are instantiating our videoprocessor class  with the model and class colors. this happens outside
#the while loop because we only need to do it once
processor = VideoProcessor(model, class_colors)
#frame counter so that we only display the first frame for our matplotlib window
frame_counter = 0
while True:
    #cap.read() gives us 2 pieces of information. a ret flag which tells us if the video is still going
    #and the frame which is the actual frame that we are processing
    ret, frame = cap.read()
    #so if the video is done we want to close the file. we want this check first
    if not ret:
        print("Finished reading video file. Exiting...")
        break
    #only if the frame counter is 0 (so only on the first frame) we want to do the following:
    if frame_counter == 0:
        #remember that whenever we use opencv (which is waht a yolo model uses) we need to 
        #convert the color to RGV if we want to use matplotlib
        converted_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        #plt.subplots() creates a figure and an axes on the figure
        fig, ax = plt.subplots()
        #ax.imshow() tells matplotlib the image we want to displa but doesnt actually display it
        ax.imshow(converted_frame) #when you do imshow, matplotlib creates a figure and an axes and event.xdata and event.ydata are the coordinates of the click in the axes coordinates
        #here we passed the frame to the BGRHandler class so we can get the BGR values when we click on the frame
        bgr_handler = BGRHandler(frame)
        #this is where the click event is connected to the figure. matplotlib uses this syntax for tying
        #click events to the figure displayed
        fig.canvas.mpl_connect('button_press_event', bgr_handler.click_event)
        #plt.show() actually displays the figure - block=True forces it to wait until window is closed
        plt.show(block=True)
        #min max is to get the min and max BGR values of all clicked pixels while it was displayed
        min_max_bgr_values = bgr_handler.min_max_bgr_values()
        print(f"Collected {len(bgr_handler.BGR_values)} clicks")
        print(f"Min BGR: {min_max_bgr_values[0]}, Max BGR: {min_max_bgr_values[1]}")
        frame_counter += 1
    #we display the mask outside the if statement because wwe want to apply the mask to all frames
    #cv2.inrange() is used to make it so all colors in a range are white while all the colors outside 
    #of that range are black
    mask = cv2.inRange(frame, min_max_bgr_values[0], min_max_bgr_values[1])
    #cv2.bitwise_and() is used to apply the mask to the frame and makes a copy of the frame 
    #with the mask applied to it
    masked_image = cv2.bitwise_and(frame, frame, mask=mask)
    
    #we run the masked_image through our processor to get the frame with the detections and annotations
    annotated_frame = processor.process_frame(masked_image, frame)
    out.write(annotated_frame)
cap.release()
out.release()

C:\Users\Guest1\.ultralytics\ultralytics\nn\tasks.py:732: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(file, map_location="cpu")


BGR values: [array([ 83, 140,  99], dtype=uint8)]
Pixel values: [ 83 140  99]
BGR values: [array([ 83, 140,  99], dtype=uint8), array([ 69, 134,  90], dtype=uint8)]
Pixel values: [ 69 134  90]
BGR values: [array([ 83, 140,  99], dtype=uint8), array([ 69, 134,  90], dtype=uint8), array([ 58, 122,  80], dtype=uint8)]
Pixel values: [ 58 122  80]
BGR values: [array([ 83, 140,  99], dtype=uint8), array([ 69, 134,  90], dtype=uint8), array([ 58, 122,  80], dtype=uint8), array([ 80, 135,  94], dtype=uint8)]
Pixel values: [ 80 135  94]
BGR values: [array([ 83, 140,  99], dtype=uint8), array([ 69, 134,  90], dtype=uint8), array([ 58, 122,  80], dtype=uint8), array([ 80, 135,  94], dtype=uint8), array([ 55, 121,  81], dtype=uint8)]
Pixel values: [ 55 121  81]
BGR values: [array([ 83, 140,  99], dtype=uint8), array([ 69, 134,  90], dtype=uint8), array([ 58, 122,  80], dtype=uint8), array([ 80, 135,  94], dtype=uint8), array([ 55, 121,  81], dtype=uint8), array([ 72, 127,  91], dtype=uint8)]
Pixel